# Multi-Agent Document Compliance System with AutoGen

In this notebook, you'll learn how to build a **multi-agent AI system** using Microsoft's [AutoGen](https://github.com/microsoft/autogen) framework.

### What you'll learn
- What an AI agent is and why we use multiple agents
- How to configure agents with different roles and instructions
- How agents collaborate in a group chat
- How to build a real-world document compliance workflow

### Scenario
Imagine you work at a company with hundreds of internal documents (policies, manuals, guidelines). Keeping them all compliant and up-to-date is tedious. We'll build an AI system with **three specialist agents** that work together to review documents automatically:

| Agent | Role |
|---|---|
| `DocumentScanner` | Reads and categorises documents |
| `ComplianceChecker` | Finds issues and rates their severity |
| `ReviewCoordinator` | Routes documents to the right people |

---

## Step 1 — Install & Import Libraries

We need two packages:
- **`pyautogen`** — the AutoGen multi-agent framework
- **`python-dotenv`** — loads secret API keys from a `.env` file so they're never hardcoded

Run the cell below once to install them (the `!` prefix runs a shell command from inside a notebook).

In [ ]:
# Uncomment the line below if you haven't installed these packages yet
# !pip install pyautogen python-dotenv

In [ ]:
import autogen
from typing import Dict, List
import json
from datetime import datetime
import os
from dotenv import load_dotenv

load_dotenv()  # Reads OPENAI_API_KEY from a .env file in the same directory

print("Libraries loaded successfully!")
print(f"AutoGen version: {autogen.__version__}")

## Step 2 — Configure the Language Model

Every agent needs a **language model (LLM)** to think and respond. We tell AutoGen which model to use via a configuration dictionary.

### Key concepts
- `config_list` — a list of model configurations (AutoGen supports fallback to the next entry if one fails)
- `llm_config` — wraps `config_list` and can hold extra settings like `temperature`

> **Why a list?** Having multiple configs lets AutoGen automatically retry with a different model if the first one hits a rate limit or error — useful in production.

In [ ]:
config_list = [
    {
        "model": "gpt-4o-mini",
        "api_key": os.environ.get("OPENAI_API_KEY"),
        "max_completion_tokens": 3000,
    }
]

llm_config = {
    "config_list": config_list,
}

# Verify the API key is loaded (prints True/False, never the key itself)
print("API key loaded:", bool(os.environ.get("OPENAI_API_KEY")))

> **Checkpoint:** If the cell above prints `API key loaded: False`, create a file named `.env` in the same folder as this notebook and add:
> ```
> OPENAI_API_KEY=sk-...
> ```

## Step 3 — Define Sample Documents

Our agents need something to analyse. Here we create three fictional company documents that each contain deliberate compliance problems — missing signatures, outdated references, and incomplete sections.

In a real system, these would be loaded from a database or file system.

In [ ]:
SAMPLE_DOCUMENTS = [
    {
        "id": "DOC001",
        "title": "Employee Remote Work Policy",
        "type": "HR Policy",
        "content": """
        Remote Work Policy
        Effective Date: January 2023
        1. Eligibility: All full-time employees
        2. Equipment: Company will provide laptop
        3. Work Hours: Standard 9-5 EST
        Note: This policy supersedes the 2021 remote work guidelines.
        """,
        "last_updated": "2023-01-15"
    },
    {
        "id": "DOC002",
        "title": "Data Security Manual",
        "type": "Technical Manual",
        "content": """
        Data Security Manual
        Version: 2.1
        1. Password Requirements: Minimum 8 characters
        2. Encryption: Use AES-256 for sensitive data
        Missing sections:
        - Incident Response Procedures
        - Data Retention Policy
        """,
        "last_updated": "2024-03-20"
    },
    {
        "id": "DOC003",
        "title": "Customer Service Guidelines",
        "type": "Operational Guidelines",
        "content": """
        Customer Service Guidelines
        1. Response Time: Within 24 hours
        2. Escalation: After 48 hours to supervisor
        3. Communication: Use template responses from 2019 handbook
        Approved by: [Missing Signature]
        Review Date: [Not Specified]
        """,
        "last_updated": "2024-11-10"
    }
]

print(f"Loaded {len(SAMPLE_DOCUMENTS)} documents")
for doc in SAMPLE_DOCUMENTS:
    print(f"  - [{doc['id']}] {doc['title']} ({doc['type']})")

### Exercise 3.1
Add a fourth document of your own to `SAMPLE_DOCUMENTS`. Give it a new `id`, `title`, `type`, and `content` with at least one deliberate compliance problem.

## Step 4 — Create the Specialist Agents

This is the heart of AutoGen. We create three types of objects:

### `AssistantAgent`
An AI-powered agent. You give it a **`system_message`** which defines its role, personality, and how it should format its responses. Think of it as a job description for the AI.

### `UserProxyAgent`  
Represents the human in the loop. It can:
- Send the opening message that kicks off the workflow
- Execute code on behalf of the human
- Ask the human for input at key moments

### `GroupChat` + `GroupChatManager`
These let multiple agents talk to each other in a shared conversation. The `GroupChatManager` decides which agent should speak next.

In [ ]:
# Agent 1: Reads and categorises documents
document_scanner = autogen.AssistantAgent(
    name="DocumentScanner",
    llm_config=llm_config,
    system_message="""You are a Document Scanner agent responsible for:
    1. Monitoring document repositories for new or updated documents
    2. Extracting document metadata (title, type, last updated date)
    3. Preparing documents for compliance analysis
    4. Identifying document categories (HR, Technical, Operational)
    Format your findings clearly with document ID, type, and key metadata."""
)

# Agent 2: Finds compliance problems and rates their severity
compliance_checker = autogen.AssistantAgent(
    name="ComplianceChecker",
    llm_config=llm_config,
    system_message="""You are a Compliance Checker agent responsible for:
    1. Analyzing documents for compliance issues such as:
       - Missing required sections (approvals, dates, signatures)
       - Outdated references or superseded content
       - Inconsistent terminology or formatting
       - Version control issues
    2. Categorizing issues by severity (Critical, Major, Minor)
    3. Providing specific recommendations for remediation
    Always structure your analysis with: Issue Type | Severity | Details | Recommendation"""
)

# Agent 3: Routes findings to the right people with deadlines
review_coordinator = autogen.AssistantAgent(
    name="ReviewCoordinator",
    llm_config=llm_config,
    system_message="""You are a Review Coordinator agent responsible for:
    1. Processing compliance check results
    2. Determining routing based on severity:
       - Critical: Immediate escalation to Legal/Executive review
       - Major: Route to department head for approval
       - Minor: Send revision request to document owner
    3. Generating action items with deadlines
    4. Creating audit trail for compliance tracking
    Format routing decisions with: Document | Issues | Routing Decision | Action Items | Due Date"""
)

# The UserProxyAgent kicks off the workflow and stops it when done
# human_input_mode="TERMINATE" means it only asks the human when the workflow ends
user_proxy = autogen.UserProxyAgent(
    name="UserProxy",
    human_input_mode="TERMINATE",
    max_consecutive_auto_reply=2,
    code_execution_config=False,
)

print("All agents created successfully!")

### Understanding `human_input_mode`

The `human_input_mode` parameter on `UserProxyAgent` controls when the human is asked for input:

| Value | Behaviour |
|---|---|
| `"NEVER"` | Fully autonomous — never pauses for human input |
| `"TERMINATE"` | Only asks at the end, when an agent says `TERMINATE` |
| `"ALWAYS"` | Pauses after every agent message for human review |

### Exercise 4.1
Change `human_input_mode` to `"ALWAYS"` and re-run the demo in Step 5. What changes? Then change it back to `"TERMINATE"`.

## Step 5 — Set Up the Group Chat

Now we wire the agents together. `GroupChat` holds the list of agents and their shared message history. `GroupChatManager` is the orchestrator — it reads the conversation so far and decides which agent should respond next.

In [ ]:
group_chat = autogen.GroupChat(
    agents=[document_scanner, compliance_checker, review_coordinator, user_proxy],
    messages=[],      # starts empty; agents fill this as they talk
    max_round=10      # safety limit — stops after 10 back-and-forth exchanges
)

manager = autogen.GroupChatManager(
    groupchat=group_chat,
    llm_config=llm_config
)

print("Group chat configured with agents:")
for agent in group_chat.agents:
    print(f"  - {agent.name}")

### What does `max_round` do?

Without a limit, agents could theoretically keep talking to each other forever. `max_round=10` means the conversation stops after 10 total agent turns, even if no agent has said `TERMINATE`. In production you'd tune this based on the complexity of the task.

## Step 6 — Run the Compliance Workflow

Everything comes together here. We:
1. Format the documents into a readable summary
2. Write an **initial message** that tells the agents what to do and in what order
3. Call `user_proxy.initiate_chat()` to start the multi-agent conversation

> **Note:** This cell makes real API calls and will take 30–90 seconds to complete.

In [ ]:
# Build a readable summary of all documents to pass to the agents
doc_summary = "\n\n".join([
    f"Document ID: {doc['id']}\n"
    f"Title: {doc['title']}\n"
    f"Type: {doc['type']}\n"
    f"Last Updated: {doc['last_updated']}\n"
    f"Content Preview:\n{doc['content'][:200]}..."
    for doc in SAMPLE_DOCUMENTS
])

# The initial message acts as the brief for the whole team
initial_message = f"""
New documents have been detected in the repository. Please process these documents through our compliance workflow:

{doc_summary}

Workflow Steps:
1. DocumentScanner: Analyse and categorise the documents
2. ComplianceChecker: Perform compliance analysis on each document
3. ReviewCoordinator: Determine routing and create action items

Begin the analysis.
"""

print("Starting multi-agent compliance workflow...")
print("=" * 60)

user_proxy.initiate_chat(
    manager,
    message=initial_message
)

### Reflection — What just happened?

Take a moment to scroll through the output above and answer:

1. Which agent spoke first? Was that the order you expected?
2. Did any agent refer to what a previous agent had said?
3. How did the `ReviewCoordinator` format its routing decisions?
4. How many rounds did the conversation take before finishing?

This back-and-forth between agents — where each builds on the previous output — is the core idea behind multi-agent systems.

## Step 7 — Generate a Compliance Report

After the agents finish, we print a human-readable summary. In a real system this might be written to a database or emailed to a manager.

In [ ]:
def generate_compliance_report(documents):
    print("\n" + "=" * 60)
    print("COMPLIANCE WORKFLOW SUMMARY")
    print("=" * 60)
    print(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"Documents Processed: {len(documents)}")
    print("\nKey Findings:")
    print("- Missing required sections detected in technical documentation")
    print("- Outdated references found in operational guidelines")
    print("- Signature approvals missing in customer service documents")
    print("\nWorkflow Complete")

generate_compliance_report(SAMPLE_DOCUMENTS)

## Step 8 — Helper: Custom Compliance Rules

This function shows how you'd define **document-type-specific rules** in a real system. Different document categories have different requirements — for example, a Technical Manual needs an Incident Response section, but an HR Policy does not.

These rules could be passed into the `ComplianceChecker`'s system message to make its analysis more precise.

In [ ]:
def create_custom_compliance_rules() -> dict:
    """Returns required sections, max document age, and signature requirements per document type."""
    return {
        "HR Policy": {
            "required_sections": ["Effective Date", "Eligibility", "Approval"],
            "max_age_months": 12,
            "requires_signature": True
        },
        "Technical Manual": {
            "required_sections": ["Version", "Security Requirements", "Incident Response"],
            "max_age_months": 6,
            "requires_signature": False
        },
        "Operational Guidelines": {
            "required_sections": ["Procedures", "Escalation", "Review Date"],
            "max_age_months": 12,
            "requires_signature": True
        }
    }

rules = create_custom_compliance_rules()
print(json.dumps(rules, indent=2))

### Exercise 8.1
Modify the `ComplianceChecker` agent's `system_message` to include these rules, so that it checks each document against the correct required sections for its type. Re-run Step 5 (recreate the group chat) and Step 6 to see how the output changes.

## Step 9 — Going Further

Here are some ideas to extend this project:

### Beginner
- Add a fourth agent (`NotificationAgent`) whose job is to draft email notifications for document owners
- Change `max_round` to `5` and observe how the output changes

### Intermediate
- Load real documents from `.txt` or `.pdf` files on disk instead of using hardcoded strings
- Store the compliance report output to a `.json` file using Python's `json` module

### Advanced
- Use a `ConversableAgent` with a custom `reply_func` to enforce the agent speaking order strictly
- Add a tool (function) that the `ReviewCoordinator` can call to look up who owns each document type

---

## Key Takeaways

| Concept | What it means |
|---|---|
| **Agent** | An AI with a specific role defined by its `system_message` |
| **`AssistantAgent`** | An AI-powered agent that responds using an LLM |
| **`UserProxyAgent`** | Represents the human; starts conversations and can run code |
| **`GroupChat`** | A shared space where multiple agents exchange messages |
| **`GroupChatManager`** | The orchestrator that decides which agent speaks next |
| **`system_message`** | The agent's job description — shapes its personality and output format |
| **`max_round`** | Safety limit on how many turns agents can take |
| **`human_input_mode`** | Controls when (if ever) the human is asked to intervene |